Use `"scikit-learn<1.6"`

In [11]:
from sentence_transformers import SentenceTransformer, models
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sentencepiece import sentencepiece_model_pb2

In [12]:
BASE_DIR = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data")
INPUT_CSV = BASE_DIR / "top300_filtered.csv"
OUTPUT_CSV = BASE_DIR / "top300_filtered_with_topics_kobert.csv"
ARTICLES_DIR = BASE_DIR

In [13]:
# Parameters are from BERTopic.ipynb best results. Parameters to be edited in later date, to work best for KoBERT.
UMAP_PARAMS = {
    'n_neighbors': 7,
    'n_components': 8,
    'min_dist': 0.08,
    'random_state': 119
}

In [14]:
HDBSCAN_PARAMS = {
    'min_cluster_size': 15,
    'min_samples': 2,
    'cluster_selection_method': 'eom'
}

In [15]:
def load_and_preprocess_documents_raw(csv_path, articles_dir):
    """
    Loads metadata and reads article files.
    Returns RAW text because KoBERT handles its own tokenization.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")

    df = pd.read_csv(csv_path)
    processed_docs = []
    valid_indices = []

    print("Processing documents (Loading Raw Text)...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        file_name = Path(row['file_path']).name
        full_path = articles_dir / file_name

        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # We use Okt here. KoBERT needs natural sentences.
            text = " ".join(text.split())
            
            processed_docs.append(text)
            valid_indices.append(idx)
            
        except Exception as e:
            print(f"Warning: Could not read file {file_name}. Error: {e}")

    df_clean = df.loc[valid_indices].copy()
    return df_clean, processed_docs

In [16]:
def run_topic_modeling_kobert():
    df, documents = load_and_preprocess_documents_raw(INPUT_CSV, ARTICLES_DIR)

    print("Initializing SKT KoBERT model...")
    # KoBERT is a base model, so we must add a pooling layer to make it an embedding model
    # We use 'skt/kobert-base-v1' or the community port 'monologg/kobert' (often more stable on HF)
    model_name = "skt/kobert-base-v1" 
    
    try:
        # Define the transformer layer
        word_embedding_model = models.Transformer(model_name, max_seq_length=512)
        
        # Define the pooling layer (Mean pooling is standard for clustering)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
        
        # Combine into a SentenceTransformer
        embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    except Exception as e:
        print(f"Error loading SKT KoBERT: {e}")
        print("Falling back to 'monologg/kobert' (Community port of SKT KoBERT)...")
        
        # FIX: Add model_args and tokenizer_args with trust_remote_code=True
        word_embedding_model = models.Transformer(
            "monologg/kobert", 
            max_seq_length=512,
            model_args={"trust_remote_code": True},
            tokenizer_args={"trust_remote_code": True}
        )
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
        embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

    # Generate Embeddings first to cache them
    print("Embedding documents with KoBERT...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True, batch_size=4)

    umap_model = UMAP(**UMAP_PARAMS)
    hdbscan_model = HDBSCAN(**HDBSCAN_PARAMS)

    print("Fitting BERTopic model...")
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        embedding_model=embedding_model, # Pass the custom KoBERT model here
        verbose=True
    )
    
    topics, probs = topic_model.fit_transform(documents, embeddings)

    df['topic'] = topics
    
    unique_topics = set(topics)
    num_topics = len(unique_topics) - 1 if -1 in unique_topics else len(unique_topics)
    print(f"Success! Found {num_topics} topics.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved to: {OUTPUT_CSV}")

    return topic_model, df

In [17]:
topic_model, final_df = run_topic_modeling_kobert()

Processing documents (Loading Raw Text)...


  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:00<00:00, 3251.31it/s]

Initializing SKT KoBERT model...


Error loading SKT KoBERT: Converting from SentencePiece and Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast converters: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Toke

...

Embedding documents with KoBERT...


Batches: 100%|██████████| 75/75 [02:30<00:00,  2.01s/it]
2026-02-14 22:06:18,827 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic model...


2026-02-14 22:06:44,353 - BERTopic - Dimensionality - Completed ✓
2026-02-14 22:06:44,356 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-14 22:06:44,485 - BERTopic - Cluster - Completed ✓
2026-02-14 22:06:44,520 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-14 22:06:44,952 - BERTopic - Representation - Completed ✓


Success! Found 6 topics.
Results saved to: C:\Users\WINDOWS 11\Desktop\kpop_agenda\data\top300_filtered_with_topics_kobert.csv


In [18]:
print("Raw Counts from DataFrame")
topic_counts = final_df['topic'].value_counts().sort_values(ascending=False)

print(f"{'Topic ID':<10} | {'Article Count'}")
print("-" * 25)
for topic_id, count in topic_counts.items():
    print(f"{topic_id:<10} | {count}")

Raw Counts from DataFrame
Topic ID   | Article Count
-------------------------
0          | 81
1          | 78
2          | 53
3          | 28
4          | 24
5          | 20
-1         | 15
